In [1]:
import os
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values

# ─────────────────────────────────────────
# 1. Configuration
# ─────────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "dreamhomes",
    "user":     "postgres",
    "password": "123",
}

CSV_DIR = "/Users/mengjieliu/sql/dreamhomes_raw_csv_v2"


# ─────────────────────────────────────────
# 2. Helper functions
# ─────────────────────────────────────────
def connect():
    return psycopg2.connect(**DB_CONFIG)


def load_csv(filename):
    path = os.path.join(CSV_DIR, filename)

    df = pd.read_csv(path, dtype=str)

    df = df.replace({
        np.nan: None,
        "nan": None,
        "NaN": None,
        "None": None,
        "": None,
        "NULL": None,
        "null": None
    })

    print(f"  Loaded {filename}: {len(df)} rows")
    return df


def to_bool(val):
    if val is None:
        return None
    return str(val).strip().lower() in ("true", "1", "yes")


def to_int(val):
    if val is None:
        return None
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return None


def to_float(val):
    if val is None:
        return None
    try:
        return float(val)
    except (ValueError, TypeError):
        return None


def clean_date(val):
    if val is None:
        return None
    if str(val).strip() in ("", "None", "nan", "NaN", "NULL", "null"):
        return None
    return val


# ─────────────────────────────────────────
# 3. Table-level import functions
# ─────────────────────────────────────────
def load_offices(conn, df):
    rows = []
    for i, r in enumerate(df.itertuples(index=False), start=1):
        rows.append((
            i,
            r.office_name,
            r.office_state,
            r.office_city,
            r.office_street_address,
            r.office_zip_code,
            r.office_phone_number,
            to_float(r.office_rent),
        ))

    sql = """
        INSERT INTO Offices
            (office_id, office_name, state, city, street_address, zip_code, phone_number, office_rent)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    print(f"  → Offices: inserted {len(rows)} rows")
    return {r[1]: r[0] for r in rows}


def load_neighbourhoods(conn, df):
    neigh_df = df[
        [
            "neighbourhood_name",
            "school_zone",
            "is_near_public_transit",
            "is_pet_friendly",
            "has_children_playground"
        ]
    ].drop_duplicates(subset=["neighbourhood_name"]).reset_index(drop=True)

    rows = []
    name_to_id = {}

    for i, r in enumerate(neigh_df.itertuples(index=False), start=1):
        rows.append((
            i,
            r.neighbourhood_name,
            r.school_zone,
            to_bool(r.is_near_public_transit),
            to_bool(r.is_pet_friendly),
            to_bool(r.has_children_playground),
        ))
        name_to_id[r.neighbourhood_name] = i

    sql = """
        INSERT INTO Neighbourhoods
            (neighbourhood_id, neighbourhood_name, school_zone,
             is_near_public_transit, is_pet_friendly, has_children_playground)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    print(f"  → Neighbourhoods: inserted {len(rows)} rows")
    return name_to_id


def load_clients(conn, df):
    rows = []
    email_to_id = {}

    for i, r in enumerate(df.itertuples(index=False), start=1):
        rows.append((
            i,
            r.first_name,
            r.last_name,
            r.email,
            r.phone,
            clean_date(r.registration_date),
        ))
        email_to_id[r.email] = i

    sql = """
        INSERT INTO Clients
            (client_id, first_name, last_name, email, phone, registration_date)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    print(f"  → Clients: inserted {len(rows)} rows")
    return email_to_id


def load_agents(conn, df, office_name_to_id):
    rows = []
    email_to_id = {}

    for i, r in enumerate(df.itertuples(index=False), start=1):
        office_id = office_name_to_id.get(r.office_name)

        rows.append((
            i,
            office_id,
            r.agent_first_name,
            r.agent_last_name,
            r.agent_email,
            r.agent_phone,
            clean_date(r.agent_hire_date),
            r.agent_license_number,
            to_float(r.agent_base_salary),
        ))

        email_to_id[r.agent_email] = i

    sql = """
        INSERT INTO Agents
            (agent_id, office_id, first_name, last_name, email, phone,
             hire_date, license_number, base_salary)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    print(f"  → Agents: inserted {len(rows)} rows")
    return email_to_id


def load_properties(conn, df, neigh_name_to_id):
    rows = []
    address_to_id = {}

    for i, r in enumerate(df.itertuples(index=False), start=1):
        neigh_id = neigh_name_to_id.get(r.neighbourhood_name)

        rows.append((
            i,
            r.property_type,
            r.property_street_address,
            r.property_city,
            r.property_state,
            neigh_id,
            to_int(r.bedrooms),
            to_int(r.bathrooms),
            to_int(r.square_feet),
            to_int(r.year_built),
            "Available",
        ))

        address_to_id[r.property_full_address] = i

    sql = """
        INSERT INTO Properties
            (property_id, property_type, street_address, city, state,
             neighbourhood_id, bedrooms, bathrooms, square_feet, year_built, current_status)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    print(f"  → Properties: inserted {len(rows)} rows")
    return address_to_id


def load_appointments(conn, df_appts, office_name_to_id, prop_addr_to_id,
                      agent_email_to_id, client_email_to_id):
    appt_rows = []
    interaction_rows = []

    appt_id = 1
    interaction_id = 1

    for r in df_appts.itertuples(index=False):
        office_id = office_name_to_id.get(r.office_name)
        prop_id = prop_addr_to_id.get(r.property_full_address)

        if office_id is None or prop_id is None:
            continue

        appt_rows.append((
            appt_id,
            office_id,
            prop_id,
            r.appointment_type,
            r.appointment_status,
            r.appointment_notes,
        ))

        agent_emails = [e.strip() for e in str(r.agent_emails).split(";") if e.strip()]
        client_emails = [e.strip() for e in str(r.client_emails).split(";") if e.strip()]

        for ae in agent_emails:
            a_id = agent_email_to_id.get(ae)

            if a_id is None:
                continue

            for ce in client_emails:
                c_id = client_email_to_id.get(ce)

                if c_id is None:
                    continue

                interaction_rows.append((
                    interaction_id,
                    appt_id,
                    a_id,
                    c_id,
                    clean_date(r.interaction_datetime),
                ))

                interaction_id += 1

        appt_id += 1

    with conn.cursor() as cur:
        if appt_rows:
            execute_values(cur, """
                INSERT INTO Appointments
                    (appointment_id, office_id, property_id, appointment_type, status, notes)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, appt_rows)

        if interaction_rows:
            execute_values(cur, """
                INSERT INTO AppointmentInteractions
                    (interaction_id, appointment_id, agent_id, client_id, interaction_datetime)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, interaction_rows)

    conn.commit()
    print(f"  → Appointments: inserted {len(appt_rows)} rows")
    print(f"  → AppointmentInteractions: inserted {len(interaction_rows)} rows")


def load_openhouses(conn, df_oh, agent_email_to_id, prop_addr_to_id):
    rows = []

    for i, r in enumerate(df_oh.itertuples(index=False), start=1):
        agent_id = agent_email_to_id.get(r.hosting_agent_email)
        prop_id = prop_addr_to_id.get(r.property_full_address)

        if agent_id is None or prop_id is None:
            continue

        rows.append((
            i,
            agent_id,
            prop_id,
            clean_date(r.start_time),
            clean_date(r.end_time),
            to_float(r.cost),
            r.notes,
        ))

    with conn.cursor() as cur:
        if rows:
            execute_values(cur, """
                INSERT INTO OpenHouses
                    (open_house_id, hosting_agent_id, property_id,
                     start_time, end_time, cost, notes)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, rows)

    conn.commit()
    print(f"  → OpenHouses: inserted {len(rows)} rows")


def load_transactions(conn, df_tx, prop_addr_to_id,
                      agent_email_to_id, client_email_to_id):
    tx_groups = {}

    for r in df_tx.itertuples(index=False):
        ref = r.transaction_reference

        if ref not in tx_groups:
            tx_groups[ref] = {
                "meta": r,
                "participants": []
            }

        tx_groups[ref]["participants"].append(r)

    listing_rows = []
    tx_rows = []
    tx_agent_rows = []
    tx_party_rows = []
    commission_rows = []
    expense_rows = []

    status_map = {
        "sold":             ("Closed",    "Completed"),
        "pending sale":     ("Pending",   "Pending"),
        "cancelled sale":   ("Cancelled", "Cancelled"),
        "expired":          ("Expired",   "Cancelled"),
        "rented":           ("Closed",    "Completed"),
        "pending rental":   ("Pending",   "Pending"),
        "cancelled rental": ("Cancelled", "Cancelled"),
    }

    listing_id = 1
    tx_id = 1
    expense_id = 1

    for ref, group in tx_groups.items():
        meta = group["meta"]
        participants = group["participants"]

        prop_id = prop_addr_to_id.get(meta.property_full_address)

        if prop_id is None:
            continue

        listing_agent_id = None

        for p in participants:
            if p.participant_type == "Agent" and p.participant_role in ("Seller Agent", "Landlord Agent"):
                listing_agent_id = agent_email_to_id.get(p.participant_email)
                break

        if listing_agent_id is None:
            for p in participants:
                if p.participant_type == "Agent":
                    listing_agent_id = agent_email_to_id.get(p.participant_email)
                    break

        raw_status = str(meta.status).strip().lower() if meta.status else "expired"
        listing_status, tx_status = status_map.get(raw_status, ("Expired", "Cancelled"))

        listing_rows.append((
            listing_id,
            prop_id,
            listing_agent_id,
            meta.transaction_type,
            clean_date(meta.transaction_start_date),
            clean_date(meta.closing_date),
            listing_status,
        ))

        tx_rows.append((
            tx_id,
            listing_id,
            meta.transaction_type,
            tx_status,
            clean_date(meta.date_of_transaction),
            clean_date(meta.closing_date),
            to_float(meta.final_price),
            ref,
        ))

        seen_agent_roles = set()
        seen_client_roles = set()

        for p in participants:
            if p.participant_type == "Agent":
                a_id = agent_email_to_id.get(p.participant_email)

                if a_id and (a_id, p.participant_role) not in seen_agent_roles:
                    seen_agent_roles.add((a_id, p.participant_role))
                    tx_agent_rows.append((tx_id, a_id, p.participant_role))

                    comm = to_float(p.commission_amount)

                    if comm is not None:
                        commission_rows.append((a_id, tx_id, comm))

            elif p.participant_type == "Client":
                c_id = client_email_to_id.get(p.participant_email)

                if c_id and (c_id, p.participant_role) not in seen_client_roles:
                    seen_client_roles.add((c_id, p.participant_role))
                    tx_party_rows.append((tx_id, c_id, p.participant_role))

        if tx_status == "Completed":
            legal = to_float(meta.legal_fees)
            other = to_float(meta.other_expenses)

            if legal is not None or other is not None:
                expense_rows.append((
                    expense_id,
                    tx_id,
                    0 if legal is None else legal,
                    0 if other is None else other,
                ))

                expense_id += 1

        listing_id += 1
        tx_id += 1

    with conn.cursor() as cur:
        if listing_rows:
            execute_values(cur, """
                INSERT INTO Listings
                    (listing_id, property_id, agent_id, listing_type,
                     start_date, end_date, listing_status)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, listing_rows)

        if tx_rows:
            execute_values(cur, """
                INSERT INTO Transactions
                    (transaction_id, listing_id, transaction_type, transaction_status,
                     contract_date, closing_date, final_price, notes)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, tx_rows)

        if tx_agent_rows:
            execute_values(cur, """
                INSERT INTO TransactionAgents
                    (transaction_id, agent_id, agent_role)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, tx_agent_rows)

        if tx_party_rows:
            execute_values(cur, """
                INSERT INTO TransactionParties
                    (transaction_id, client_id, role_type)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, tx_party_rows)

        if commission_rows:
            execute_values(cur, """
                INSERT INTO Commissions
                    (agent_id, transaction_id, commission_amount)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, commission_rows)

        if expense_rows:
            execute_values(cur, """
                INSERT INTO Expenses
                    (expense_id, transaction_id, legal_fees, other_expenses)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, expense_rows)

    conn.commit()

    print(f"  → Listings: inserted {len(listing_rows)} rows")
    print(f"  → Transactions: inserted {len(tx_rows)} rows")
    print(f"  → TransactionAgents: inserted {len(tx_agent_rows)} rows")
    print(f"  → TransactionParties: inserted {len(tx_party_rows)} rows")
    print(f"  → Commissions: inserted {len(commission_rows)} rows")
    print(f"  → Expenses: inserted {len(expense_rows)} rows")


def load_property_owners(conn, df_tx, prop_addr_to_id, client_email_to_id):
    """
    Builds PropertyOwners from raw_transactions.csv.

    Logic:
    - Seller/Landlord clients are treated as owners.
    - For completed sales, sellers' ownership ends at closing_date.
    - For completed sales, buyers become current owners starting at closing_date.
    - For rentals, landlords remain owners.
    - Tenants/renters are not inserted as property owners.
    - For expired/cancelled listings, seller/landlord remains current owner.
    """

    tx_groups = {}

    for r in df_tx.itertuples(index=False):
        ref = r.transaction_reference

        if ref not in tx_groups:
            tx_groups[ref] = {
                "meta": r,
                "participants": []
            }

        tx_groups[ref]["participants"].append(r)

    owner_rows = []
    property_owner_id = 1

    seller_roles = {"Seller", "Landlord"}
    buyer_roles = {"Buyer"}
    completed_sale_statuses = {"sold"}

    seen_ownership_records = set()

    for ref, group in tx_groups.items():
        meta = group["meta"]
        participants = group["participants"]

        prop_id = prop_addr_to_id.get(meta.property_full_address)

        if prop_id is None:
            continue

        raw_status = str(meta.status).strip().lower() if meta.status else "expired"

        transaction_start_date = clean_date(meta.transaction_start_date)
        closing_date = clean_date(meta.closing_date)
        date_of_transaction = clean_date(meta.date_of_transaction)

        transfer_date = closing_date or date_of_transaction or transaction_start_date

        for p in participants:
            if p.participant_type != "Client":
                continue

            client_id = client_email_to_id.get(p.participant_email)

            if client_id is None:
                continue

            role = p.participant_role

            if role in seller_roles:
                if raw_status in completed_sale_statuses:
                    ownership_end_date = transfer_date
                    is_current_owner = False
                else:
                    ownership_end_date = None
                    is_current_owner = True

                record_key = (
                    prop_id,
                    client_id,
                    transaction_start_date,
                    ownership_end_date,
                    is_current_owner
                )

                if record_key not in seen_ownership_records:
                    seen_ownership_records.add(record_key)

                    owner_rows.append((
                        property_owner_id,
                        prop_id,
                        client_id,
                        transaction_start_date,
                        ownership_end_date,
                        is_current_owner
                    ))

                    property_owner_id += 1

            elif role in buyer_roles and raw_status in completed_sale_statuses:
                record_key = (
                    prop_id,
                    client_id,
                    transfer_date,
                    None,
                    True
                )

                if record_key not in seen_ownership_records:
                    seen_ownership_records.add(record_key)

                    owner_rows.append((
                        property_owner_id,
                        prop_id,
                        client_id,
                        transfer_date,
                        None,
                        True
                    ))

                    property_owner_id += 1

    with conn.cursor() as cur:
        if owner_rows:
            execute_values(cur, """
                INSERT INTO PropertyOwners
                    (property_owner_id, property_id, client_id,
                     ownership_start_date, ownership_end_date, is_current_owner)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, owner_rows)

    conn.commit()
    print(f"  → PropertyOwners: inserted {len(owner_rows)} rows")


# ─────────────────────────────────────────
# 4. Validation queries
# ─────────────────────────────────────────
VALIDATION_QUERIES = [
    ("Offices",                 "SELECT COUNT(*) FROM Offices"),
    ("Neighbourhoods",          "SELECT COUNT(*) FROM Neighbourhoods"),
    ("Clients",                 "SELECT COUNT(*) FROM Clients"),
    ("Agents",                  "SELECT COUNT(*) FROM Agents"),
    ("Properties",              "SELECT COUNT(*) FROM Properties"),
    ("PropertyOwners",          "SELECT COUNT(*) FROM PropertyOwners"),
    ("Appointments",            "SELECT COUNT(*) FROM Appointments"),
    ("AppointmentInteractions", "SELECT COUNT(*) FROM AppointmentInteractions"),
    ("OpenHouses",              "SELECT COUNT(*) FROM OpenHouses"),
    ("Listings",                "SELECT COUNT(*) FROM Listings"),
    ("Transactions",            "SELECT COUNT(*) FROM Transactions"),
    ("TransactionAgents",       "SELECT COUNT(*) FROM TransactionAgents"),
    ("TransactionParties",      "SELECT COUNT(*) FROM TransactionParties"),
    ("Commissions",             "SELECT COUNT(*) FROM Commissions"),
    ("Expenses",                "SELECT COUNT(*) FROM Expenses"),
]


def validate(conn):
    print("\n── Validation Results ──────────────")

    with conn.cursor() as cur:
        for label, sql in VALIDATION_QUERIES:
            cur.execute(sql)
            count = cur.fetchone()[0]
            print(f"  {label:<28} {count:>6} rows")

    print("────────────────────────────────────")


# ─────────────────────────────────────────
# 5. Main
# ─────────────────────────────────────────
def main():
    print("=== Dream Homes NYC — Data Import ===\n")

    conn = connect()
    print("✓ Database connection successful\n")

    print("[1] Loading CSV files...")

    df_offices = load_csv("raw_offices.csv")
    df_clients = load_csv("raw_clients.csv")
    df_agents = load_csv("raw_agents.csv")
    df_properties = load_csv("raw_properties.csv")
    df_appts = load_csv("raw_appointments.csv")
    df_oh = load_csv("raw_openhouses.csv")
    df_tx = load_csv("raw_transactions.csv")

    print("\n[2] Importing data...")

    office_name_to_id = load_offices(conn, df_offices)
    neigh_name_to_id = load_neighbourhoods(conn, df_properties)
    client_email_to_id = load_clients(conn, df_clients)
    agent_email_to_id = load_agents(conn, df_agents, office_name_to_id)
    prop_addr_to_id = load_properties(conn, df_properties, neigh_name_to_id)

    load_appointments(
        conn,
        df_appts,
        office_name_to_id,
        prop_addr_to_id,
        agent_email_to_id,
        client_email_to_id
    )

    load_openhouses(
        conn,
        df_oh,
        agent_email_to_id,
        prop_addr_to_id
    )

    load_transactions(
        conn,
        df_tx,
        prop_addr_to_id,
        agent_email_to_id,
        client_email_to_id
    )

    load_property_owners(
        conn,
        df_tx,
        prop_addr_to_id,
        client_email_to_id
    )

    validate(conn)

    conn.close()

    print("\n✓ All done!")


if __name__ == "__main__":
    main()

=== Dream Homes NYC — Data Import ===

✓ Database connection successful

[1] Loading CSV files...
  Loaded raw_offices.csv: 6 rows
  Loaded raw_clients.csv: 20 rows
  Loaded raw_agents.csv: 20 rows
  Loaded raw_properties.csv: 90 rows
  Loaded raw_appointments.csv: 346 rows
  Loaded raw_openhouses.csv: 53 rows
  Loaded raw_transactions.csv: 970 rows

[2] Importing data...
  → Offices: inserted 6 rows
  → Neighbourhoods: inserted 18 rows
  → Clients: inserted 20 rows
  → Agents: inserted 20 rows
  → Properties: inserted 90 rows
  → Appointments: inserted 346 rows
  → AppointmentInteractions: inserted 596 rows
  → OpenHouses: inserted 53 rows
  → Listings: inserted 228 rows
  → Transactions: inserted 228 rows
  → TransactionAgents: inserted 456 rows
  → TransactionParties: inserted 514 rows
  → Commissions: inserted 342 rows
  → Expenses: inserted 171 rows
  → PropertyOwners: inserted 328 rows

── Validation Results ──────────────
  Offices                           6 rows
  Neighbourhoo